In [2]:
from bs4 import BeautifulSoup
import re

In [4]:
# sample
# pipelagging type 4
true=True
sample3 = {
  "price": {
    "label": "Ex. VAT £2.37 £1.94",
    "tag": "div",
    "type": "",
    "name": "",
    "id": "",
    "class_name": "final-price-excl-tax pr-3",
    "bbox": {
      "left": 906.890625,
      "top": 642.75,
      "right": 997.765625,
      "bottom": 698.75
    }
  },
  "inputs": [
    {
      "label": "Insulation to Suit Pipe Size",
      "group_label": "Options",
      "tag": "select",
      "type": "",
      "name": "super_attribute[186]",
      "id": "attribute186",
      "bbox": {
        "left": 1070.5625,
        "top": 774.75,
        "right": 1226.234375,
        "bottom": 818.75
      },
      "is_price_relevant": true
    },
    {
      "label": "Insulation Thickness",
      "group_label": "Options",
      "tag": "select",
      "type": "",
      "name": "super_attribute[185]",
      "id": "attribute185",
      "bbox": {
        "left": 1070.5625,
        "top": 841.75,
        "right": 1226.234375,
        "bottom": 885.75
      },
      "is_price_relevant": true
    }
  ]
}
# remove duplicates
input_sample = sample3
def remove_duplicates_by_label(sample):
    seen_labels = set()
    unique_inputs = []

    for inp in sample["inputs"]:
        label = inp.get("label", "").strip().lower()

        if label not in seen_labels:
            seen_labels.add(label)
            unique_inputs.append(inp)

    sample["inputs"] = unique_inputs
    return sample
sample_deduplicated = remove_duplicates_by_label(input_sample)

# function to generate XPath based on element attributes
def get_xpath(element):
    tag = element.get("tag", "*")
    
    if element.get("id"):
        return f'//*[@id="{element["id"]}"]'
    
    name = element.get("name", "")
    type_ = element.get("type", "")
    if name and type_:
        return f'//{tag}[@name="{name}" and @type="{type_}"]'
    elif name:
        return f'//{tag}[@name="{name}"]'
    
    class_name = element.get("class_name", "")
    if class_name:
        return f'//{tag}[contains(@class, "{class_name}")]'
    
    return f'//{tag}'

# function to generate a list of (XPath, label) tuples for all input elements
def get_xpath_map(sample: dict) -> list:
    xpath_map = []

    for elem in sample["inputs"]:
        tag       = elem.get("tag", "input")
        elem_id   = elem.get("id", "")
        elem_name = elem.get("name", "")
        elem_type = elem.get("type", "").lower()
        elem_label = elem.get("label", "")

        if not elem_id and not elem_name:
            continue

        if elem_id:
            xpath = f"//select[@id='{elem_id}']" if tag == "select" else f"//input[@id='{elem_id}']"
        else:
            xpath = f"//{tag}[@name='{elem_name}']"

        if tag == "select":         action_type = "select"
        elif elem_type == "radio":  action_type = "radio"
        elif elem_type == "checkbox": action_type = "checkbox"
        else:                       action_type = "input"

        xpath_map.append((xpath, elem_label))  # ← tuple, not string

    return xpath_map  # ← list of tuples

# html


In [6]:
from lxml import etree

def extract_elements(html: str, xpath_map: list) -> str:
    tree = etree.HTML(html)
    results = []

    for xpath, elem_label in xpath_map:
        els = tree.xpath(xpath)
        if els:
            results.append(f"label name - {elem_label}\n")
            results.append(etree.tostring(els[0], encoding="unicode"))

    return results

with open("PipeLaggingType4.html", "r", encoding="utf-8") as f:
    html = f.read()

xpath_map = get_xpath_map(input_sample)

elements = extract_elements(html, xpath_map)
print(elements)

['label name - Insulation to Suit Pipe Size\n', '<select name="super_attribute[186]" id="attribute186" class="form-select super-attribute-select max-w-full" x-on:change="changeOption(186, event.target.value)" required="">\n                    <option value="">\n                        Choose an Option...                    </option>\n                    <template x-for="(item, index) in getAllowedAttributeOptions(186)" :key="item.id"/><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="122">6 + £0.31</option><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="123">10 + £1.28</option><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="124">12</option><option :value="item.id" x-html="getAttribute

In [15]:
def _clean_label(text: str) -> str:
    """Strip price suffix from a label string."""
    return re.sub(r"\s*\+\s*[£$€¥₹][\d.,]+", "", text).strip()


def _extract_map_from_html(html: str) -> dict:
    """Parse a <select> HTML and return {clean_label: value} map."""
    soup = BeautifulSoup(html, "html.parser")
    map_ = {}
    for opt in soup.find_all("option"):
        raw_label = opt.get_text(strip=True)
        clean_label = _clean_label(raw_label)
        value = opt.get("value", "").strip()
        if clean_label and value:
            map_[clean_label] = value
    return map_


def insulation_to_suit_pipe_size_mapper(html: str, visible_text: str) -> str | None:
    map_ = _extract_map_from_html(html)
    return map_.get(_clean_label(str(visible_text)))  # clean input too


def insulation_thickness_mapper(html: str, visible_text: str) -> str | None:
    map_ = _extract_map_from_html(html)
    return map_.get(_clean_label(str(visible_text)))  # clean input too

In [7]:
elements

['label name - Insulation to Suit Pipe Size\n',
 '<select name="super_attribute[186]" id="attribute186" class="form-select super-attribute-select max-w-full" x-on:change="changeOption(186, event.target.value)" required="">\n                    <option value="">\n                        Choose an Option...                    </option>\n                    <template x-for="(item, index) in getAllowedAttributeOptions(186)" :key="item.id"/><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="122">6 + £0.31</option><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="123">10 + £1.28</option><option :value="item.id" x-html="getAttributeOptionLabel(item)" :selected="selectedValues[186] ===&#10;                                item.id" value="124">12</option><option :value="item.id" x-html="getAttribut

In [16]:
pipe_size_html = elements[1]
thickness_html = elements[3]

# Each function is bound to its own selector's HTML
print(insulation_to_suit_pipe_size_mapper(pipe_size_html, "22"))   # → "128"
print(insulation_to_suit_pipe_size_mapper(pipe_size_html, "48"))   # → "164"

print(insulation_thickness_mapper(thickness_html, "9"))            # → "165"
print(insulation_thickness_mapper(thickness_html, "32"))           # → "121"

# Full visible text with price also works
print(insulation_thickness_mapper(thickness_html, "9 + £0.34"))    # → "116"

128
164
165
121
165
